# 18 — Architecture 비교: U-Net vs FPN vs DeepLab v3+ vs MA-Net

**목적**: "왜 U-Net?" 슬라이드 보강 — 학술 근거 + **실험적 증명**

**실험 설계 (공정 비교)**:
- 동일 Encoder: ResNet-34 (ImageNet pretrained)
- 동일 Loss: Combo Loss (0.5 Dice + 0.5 CE)
- 동일 Optimizer: Adam(lr=1e-4) + CosineAnnealingLR
- 동일 학습: 10 epochs, batch 8, 5K train + 500 val
- 동일 입력: RGB (in_channels=3) — fair baseline

**비교 모델 4종 (segmentation_models_pytorch)**:
| 모델 | 클래스 | 학술 출처 |
|------|--------|----------|
| U-Net | `smp.Unet` | Ronneberger 2015 MICCAI |
| FPN | `smp.FPN` | Lin 2017 CVPR |
| DeepLab v3+ | `smp.DeepLabV3Plus` | Chen 2018 ECCV |
| MA-Net | `smp.MAnet` | Fan 2020 IEEE Access |

**예상 소요**: T4 GPU 기준 모델당 약 50분 × 4 = 약 3-4시간

**중간 끊김 대비**: 모델별 ckpt + 결과 JSON 자동 저장. 재실행 시 cached 모델은 skip.

## 1. 환경 셋업

In [ ]:
!nvidia-smi

In [ ]:
!pip install --quiet segmentation-models-pytorch albumentations datasets pyyaml

In [ ]:
import torch, segmentation_models_pytorch as smp
import numpy as np
import matplotlib.pyplot as plt
import json, os, time
from pathlib import Path
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('smp:', smp.__version__)
print('device:', device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/cv-final'
import sys
sys.path.insert(0, PROJECT_ROOT)

# 출력 경로
OUTPUT_DIR = Path(PROJECT_ROOT) / 'data' / 'outputs' / 'model_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
RESULT_JSON = OUTPUT_DIR / 'results.json'
print('output dir:', OUTPUT_DIR)

## 2. 데이터셋 (5K train + 500 val · RGB only)

In [ ]:
from project.segmentation import CelebAMaskHQDataset, get_train_transform, get_val_transform

INPUT_SIZE = 256
NUM_CLASSES = 6  # background, nose, eye, mouth, skin, unused
TRAIN_SUBSET = 5000
VAL_SUBSET = 500
BATCH_SIZE = 8

# Train
train_ds = CelebAMaskHQDataset(
    split='train',
    transform=get_train_transform(size=INPUT_SIZE, with_heatmap=False),
    subset_size=TRAIN_SUBSET,
    cache_dir='/content/data/celebamask',
    input_size=INPUT_SIZE,
    use_landmark=False,
)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Val
val_ds = CelebAMaskHQDataset(
    split='val',
    transform=get_val_transform(size=INPUT_SIZE, with_heatmap=False),
    subset_size=VAL_SUBSET,
    cache_dir='/content/data/celebamask',
    input_size=INPUT_SIZE,
    use_landmark=False,
)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} samples · {len(train_dl)} batches')
print(f'Val:   {len(val_ds)} samples · {len(val_dl)} batches')

## 3. 4 모델 빌드 함수 (공정 비교)

In [ ]:
def build_model(arch_name, num_classes=NUM_CLASSES):
    """공정 비교용 모델 빌드.
    
    모든 모델: ResNet-34 + ImageNet pretrained + in_channels=3.
    """
    common = dict(
        encoder_name='resnet34',
        encoder_weights='imagenet',
        in_channels=3,
        classes=num_classes,
    )
    if arch_name == 'unet':
        return smp.Unet(**common)
    elif arch_name == 'fpn':
        return smp.FPN(**common)
    elif arch_name == 'deeplabv3plus':
        return smp.DeepLabV3Plus(**common)
    elif arch_name == 'manet':
        return smp.MAnet(**common)
    else:
        raise ValueError(f'unknown arch: {arch_name}')


# 빌드 테스트
ARCHS = ['unet', 'fpn', 'deeplabv3plus', 'manet']
ARCH_LABELS = {
    'unet': 'U-Net (Ronneberger 2015)',
    'fpn': 'FPN (Lin 2017)',
    'deeplabv3plus': 'DeepLab v3+ (Chen 2018)',
    'manet': 'MA-Net (Fan 2020)',
}

for arch in ARCHS:
    m = build_model(arch)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'{ARCH_LABELS[arch]:<35} {n_params:>6.2f} M params')
    del m
torch.cuda.empty_cache()

## 4. 학습 + 평가 함수

In [ ]:
from project.segmentation.losses import ComboLoss
from project.segmentation.evaluation import evaluate_model_on_loader

EPOCHS = 10
LR = 1e-4

CLASS_NAMES = ['background', 'nose', 'eye', 'mouth', 'skin', 'unused']
VALID_IDX_5 = [0, 1, 2, 3, 4]  # unused 제외 (공식 보고용)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device).long()
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def train_and_eval(arch, train_dl, val_dl, epochs=EPOCHS, lr=LR, ckpt_dir=CKPT_DIR):
    """단일 architecture 학습 + 평가. 결과 dict 반환."""
    ckpt_path = ckpt_dir / f'{arch}_resnet34.pt'
    
    # 캐시된 체크포인트 활용
    if ckpt_path.exists():
        print(f'  [cached] {ckpt_path.name} 로드')
        model = build_model(arch).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    else:
        print(f'  학습 시작 (epochs={epochs}, lr={lr})')
        model = build_model(arch).to(device)
        criterion = ComboLoss(dice_weight=0.5, ce_weight=0.5)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
        
        train_history = []
        t0 = time.time()
        for ep in range(epochs):
            loss = train_one_epoch(model, train_dl, criterion, optimizer, device)
            scheduler.step()
            elapsed = time.time() - t0
            print(f'    epoch {ep+1:2d}/{epochs} · loss {loss:.4f} · '
                  f'elapsed {elapsed/60:.1f}m')
            train_history.append({'epoch': ep+1, 'loss': loss})
        
        # 저장
        torch.save(model.state_dict(), ckpt_path)
        print(f'  체크포인트 저장: {ckpt_path}')
    
    # 평가
    print(f'  평가 중 (val {len(val_dl.dataset)}장)...')
    t0 = time.time()
    res = evaluate_model_on_loader(model, val_dl, NUM_CLASSES, device=device)
    per_class = np.array(res['per_class_iou'])
    res['mIoU_5class'] = float(per_class[VALID_IDX_5].mean())
    res['n_params_M'] = sum(p.numel() for p in model.parameters()) / 1e6
    res['eval_time_sec'] = time.time() - t0
    print(f'    5-class mIoU: {res["mIoU_5class"]:.4f} '
          f'· 6-class mIoU: {res["mIoU"]:.4f} '
          f'· Params: {res["n_params_M"]:.1f}M')
    
    del model
    torch.cuda.empty_cache()
    return res

## 5. 메인 실행 — 4 모델 sequential 학습 + 평가

각 모델 약 50분 × 4 = 약 3-4시간

In [ ]:
# 기존 결과 로드 (중간 끊김 대비)
results = {}
if RESULT_JSON.exists():
    with open(RESULT_JSON, 'r') as f:
        results = json.load(f)
    print(f'기존 결과 로드: {list(results.keys())}')
else:
    print('새로 시작')

# 4 모델 sequential
for arch in ARCHS:
    if arch in results and 'mIoU_5class' in results[arch]:
        print(f'\n[skip] {arch} — 이미 평가 완료 '
              f'(5-class mIoU {results[arch]["mIoU_5class"]:.4f})')
        continue
    
    print(f'\n{"="*60}')
    print(f'모델: {ARCH_LABELS[arch]}')
    print(f'{"="*60}')
    
    t_start = time.time()
    try:
        res = train_and_eval(arch, train_dl, val_dl)
        res['arch'] = arch
        res['label'] = ARCH_LABELS[arch]
        res['total_time_min'] = (time.time() - t_start) / 60
        results[arch] = res
        
        # 매 모델마다 결과 저장 (중간 끊김 대비)
        with open(RESULT_JSON, 'w') as f:
            json.dump(results, f, indent=2, default=str)
        print(f'  결과 저장: {RESULT_JSON}')
    except Exception as e:
        print(f'  ERROR ({arch}): {e}')
        import traceback
        traceback.print_exc()

print(f'\n=== 완료: {len(results)}개 모델 ===')

## 6. 결과 비교 표

In [ ]:
import pandas as pd

rows = []
for arch, res in results.items():
    row = {
        'Model': ARCH_LABELS[arch],
        'mIoU_5class': round(res['mIoU_5class'], 4),
        'mIoU_6class': round(res['mIoU'], 4),
        'Mean Dice': round(res['mean_dice'], 4),
        'Overall Acc': round(res['overall_accuracy'], 4),
        'Params (M)': round(res.get('n_params_M', 0), 2),
        'Time (min)': round(res.get('total_time_min', 0), 1),
    }
    # per-class IoU
    for i, cname in enumerate(CLASS_NAMES):
        row[f'IoU_{cname}'] = round(res['per_class_iou'][i], 4)
    rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values('mIoU_5class', ascending=False).reset_index(drop=True)

print('=' * 80)
print('Architecture 비교 결과 (정렬: 5-class mIoU 내림차순)')
print('=' * 80)
print(df.to_string(index=False))

# CSV 저장
CSV_PATH = OUTPUT_DIR / 'architecture_comparison.csv'
df.to_csv(CSV_PATH, index=False)
print(f'\n저장: {CSV_PATH}')
df

## 7. 시각화 — 발표용 PNG 2장

In [ ]:
# PNG 1: mIoU 비교 막대 그래프
NAVY = '#1E2761'
ACCENT = '#F96167'
GOLD = '#F5A623'
GREEN = '#48BB78'
MUTED = '#7B8FA1'

fig, ax = plt.subplots(figsize=(11, 6))
labels = [ARCH_LABELS[arch].split(' (')[0] for arch in df['Model'].apply(lambda x: x.split(' (')[0])]
values_5 = df['mIoU_5class'].values
values_6 = df['mIoU_6class'].values

x = np.arange(len(labels))
width = 0.35

# U-Net은 액센트 색, 나머지는 muted
colors_5 = [ACCENT if 'U-Net' in lab else NAVY for lab in labels]
colors_6 = [GOLD if 'U-Net' in lab else MUTED for lab in labels]

bars1 = ax.bar(x - width/2, values_5, width, label='5-class mIoU (공식)', color=colors_5)
bars2 = ax.bar(x + width/2, values_6, width, label='6-class mIoU', color=colors_6, alpha=0.6)

# 값 표시
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=9, color=MUTED)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('mIoU', fontsize=12)
ax.set_title('Architecture 공정 비교 (동일 조건: ResNet-34, Combo Loss, 10 epoch, 5K)',
             fontsize=13, color=NAVY, fontweight='bold')
ax.set_ylim(0, max(values_5.max(), values_6.max()) * 1.15)
ax.legend(loc='lower right', fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
PNG_BAR = OUTPUT_DIR / 'architecture_comparison_bar.png'
plt.savefig(PNG_BAR, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {PNG_BAR}')

In [ ]:
# PNG 2: Per-class IoU 비교 (4 모델 × 5 클래스)
fig, ax = plt.subplots(figsize=(12, 6))

valid_classes = ['background', 'nose', 'eye', 'mouth', 'skin']
x = np.arange(len(valid_classes))
width = 0.2

colors_map = {
    'unet': ACCENT,
    'fpn': NAVY,
    'deeplabv3plus': GOLD,
    'manet': GREEN,
}

# 정렬: U-Net 먼저
ordered_archs = ['unet'] + [a for a in ARCHS if a != 'unet' and a in results]

for i, arch in enumerate(ordered_archs):
    if arch not in results:
        continue
    res = results[arch]
    per_class = [res['per_class_iou'][CLASS_NAMES.index(c)] for c in valid_classes]
    offset = (i - len(ordered_archs)/2 + 0.5) * width
    bars = ax.bar(x + offset, per_class, width, 
                  label=ARCH_LABELS[arch].split(' (')[0],
                  color=colors_map.get(arch, MUTED))

ax.set_xticks(x)
ax.set_xticklabels(valid_classes, fontsize=11)
ax.set_ylabel('IoU', fontsize=12)
ax.set_title('Per-class IoU 비교 (얼굴 부위별 정확도)',
             fontsize=13, color=NAVY, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
PNG_PERCLASS = OUTPUT_DIR / 'architecture_per_class_iou.png'
plt.savefig(PNG_PERCLASS, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {PNG_PERCLASS}')

## 8. 발표 슬라이드용 핵심 메시지

In [ ]:
# 1등 모델 추출
best = df.iloc[0]
second = df.iloc[1] if len(df) > 1 else None

print('=' * 70)
print('발표 슬라이드 6 (왜 U-Net?) 실험적 보강 메시지')
print('=' * 70)
print()
print(f'★ 1등: {best["Model"]}')
print(f'  5-class mIoU: {best["mIoU_5class"]:.4f}')
print(f'  Params: {best["Params (M)"]} M')
print()

if second is not None:
    diff = best['mIoU_5class'] - second['mIoU_5class']
    print(f'2등: {second["Model"]} (mIoU {second["mIoU_5class"]:.4f})')
    print(f'  → 1등과 차이: {diff*100:+.2f}%p')
    print()

print('전체 순위:')
for i, row in df.iterrows():
    medal = ['🥇', '🥈', '🥉', '4️⃣'][i] if i < 4 else f'{i+1}'
    print(f'  {medal} {row["Model"]:<35} mIoU {row["mIoU_5class"]:.4f}')

print()
if 'U-Net' in best['Model']:
    print('✅ U-Net이 가장 높음 — 학술 근거 + 실험적 증명 일치')
    print('   슬라이드 6 메시지: "동일 조건에서 U-Net이 최고 mIoU"')
else:
    print('⚠ U-Net이 1등 아님 — 정직한 보고 + 디스커션 필요')
    print('   슬라이드 6 수정: "U-Net은 trade-off (속도/단순성) 선택"')

In [ ]:
# 결과 파일 목록 (다운로드용)
print('📦 다운로드할 파일:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {f.relative_to(OUTPUT_DIR)}: {size_kb:.1f} KB')

# Zip 생성 (다운로드 편의)
import shutil
ZIP_PATH = OUTPUT_DIR.parent / 'model_comparison_results.zip'
shutil.make_archive(str(ZIP_PATH).replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f'\n📥 Zip 파일: {ZIP_PATH}')

from google.colab import files
# files.download(str(ZIP_PATH))  # 주석 해제하면 자동 다운로드

## ✅ 완료 체크리스트

- [ ] 4 모델 모두 학습 완료 (`results.json`에 4 entry)
- [ ] `architecture_comparison.csv` 저장됨
- [ ] `architecture_comparison_bar.png` 저장됨
- [ ] `architecture_per_class_iou.png` 저장됨
- [ ] `model_comparison_results.zip` 다운로드

## 📊 발표 활용

1. **슬라이드 6 (왜 U-Net?)** 보강:
   - 학술 근거 (좌측) + **실험 결과 표** (중앙) + **bar 그래프** (우측)
2. **Q&A 대비**: "왜 DeepLab 안 썼나?" → "동일 조건 실험에서 U-Net이 +X.XX%p 높음"